<a href="https://colab.research.google.com/github/luissvalencia/ZapatosWeb/blob/main/special/lite/derfuu_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os

# 🔹 Instalar dependencias
!apt -y update -qq
!apt -y install -qq aria2
!pip install -q torch==1.13.1+cu116 torchvision==0.14.1+cu116 torchaudio==0.13.1 torchtext==0.14.1 torchdata==0.5.1 --extra-index-url https://download.pytorch.org/whl/cu116 -U
!pip install -q xformers==0.0.16 triton==2.0.0 -U
!pip install -q mediapipe==0.9.1.0 addict yapf fvcore omegaconf

# 🔹 Descargar ComfyUI
!git clone -b v1.2.derfuu https://github.com/camenduru/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!git reset --hard
!git clone -b v1.2 https://github.com/camenduru/Derfuu_ComfyUI_ModdedNodes /content/ComfyUI/custom_nodes/Derfuu_ComfyUI_ModdedNodes

# 🔹 Cloudflare túnel
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64
import atexit, requests, subprocess, time, re
from random import randint
from threading import Timer
from queue import Queue

def cloudflared(port, metrics_port, output_queue):
    atexit.register(lambda p: p.terminate(), subprocess.Popen(
        ['/content/cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
    ))
    attempts, tunnel_url = 0, None
    while attempts < 10 and not tunnel_url:
        attempts += 1
        time.sleep(3)
        try:
            tunnel_url = re.search("(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text).group("url")
        except:
            pass
    if not tunnel_url:
        raise Exception("Can't connect to Cloudflare Edge")
    output_queue.put(tunnel_url)

output_queue, metrics_port = Queue(), randint(8100, 9000)
thread = Timer(2, cloudflared, args=(8188, metrics_port, output_queue))
thread.start()
thread.join()
tunnel_url = output_queue.get()
os.environ['webui_url'] = tunnel_url
print("🔗 Accede a ComfyUI en:", tunnel_url)

# 🔹 Modelos extra (upscale y VAE)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth -d /content/ComfyUI/models/upscale_models -o RealESRGAN_x2plus.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/hakurei/waifu-diffusion-v1-4/resolve/main/vae/kl-f8-anime2.ckpt -d /content/ComfyUI/models/vae -o kl-f8-anime2.ckpt

# 🔹 Descargar tu modelo actualizado de Civitai
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
"https://civitai.com/api/download/models/267954" \
-d /content/ComfyUI/models/checkpoints \
-o lazymixRealAmateur_v40.safetensors


# 🔹 Iniciar ComfyUI
!python main.py --dont-print-server


<>:34: SyntaxWarning: invalid escape sequence '\/'
<>:34: SyntaxWarning: invalid escape sequence '\/'
/tmp/ipython-input-64494192.py:34: SyntaxWarning: invalid escape sequence '\/'
  tunnel_url = re.search("(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text).group("url")


38 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
aria2 is already the newest version (1.36.0-1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
ERROR: Could not find a version that satisfies the requirement torch==1.13.1+cu116 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0)
ERROR: No matching distribution found for torch==1.13.1+cu116
  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement triton==2.0.0 (from versions: 2.2.0, 2.3.0, 2.3.1, 3.0.0, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0)
ERROR: No matching distribution found for triton==2.0.0
ERROR: Could not find a version that satisfies the requirement mediapipe==0.9.1.0 (from versions: 0.10.13, 0.10.14, 0.10.15,